# Friends Trip Planning Demo

This notebook runs a live LLM-backed group chat where friends try to decide whether to travel and, if so, where to go.

Each friend gets a private planning brief with traits, budget concerns, travel hopes, and worries. The group chat starts with one friend asking to plan the trip. The simulation ends with either a shared destination or `NO_TRIP`.

In [1]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

import api.websockets as websocket_module
from chatapp import TestClientChatGateway
from db.session import create_connection, get_db, init_db
from main import app
from simulation.runtimes.llm import resolve_llm_provider_config
from simulation.runtimes.trip_planner import TripFriendPersona, TripPlannerRuntimeFactory
from simulation.trip_planner import FriendsTripConfig, FriendsTripSimulationEngine

workspace = TemporaryDirectory()
database_path = Path(workspace.name) / "friends-trip-demo.db"
init_db(database_path)

def testing_session_local():
    return create_connection(database_path)

def override_get_db():
    db = testing_session_local()
    try:
        yield db
    finally:
        db.close()

app.dependency_overrides[get_db] = override_get_db
original_session_local = websocket_module.SessionLocal
websocket_module.SessionLocal = testing_session_local

client_manager = TestClient(app)
client = client_manager.__enter__()
gateway = TestClientChatGateway(client)
provider_config = resolve_llm_provider_config()
runtime_factory = TripPlannerRuntimeFactory.from_environment(provider_config.provider)
engine = FriendsTripSimulationEngine(gateway, runtime_factory=runtime_factory)

print(f"Demo database: {database_path}")
pprint({"provider": provider_config.provider, "model": provider_config.model, "base_url": provider_config.base_url})

Demo database: /var/folders/cm/sn2bpgln3zdc0lh8s2384hr40000gn/T/tmpah86ix85/friends-trip-demo.db
{'base_url': 'https://api.pinference.ai/api/v1',
 'model': 'meta-llama/llama-3.3-70b-instruct',
 'provider': 'primeintellect'}


In [2]:
config = FriendsTripConfig(
    admin_name="Trip Host",
    group_title="Friends Trip Planning",
    initiator_name="Nina",
    destination_options=["Lisbon", "Mexico City", "Vancouver"],
    max_discussion_rounds=2,
    friends=[
        TripFriendPersona(
            name="Nina",
            traits=["empathetic", "group glue", "likes slow mornings"],
            budget_notes="Can travel, but only if the plan still feels fair to friends with tighter budgets.",
            travel_hopes="Wants a place where everyone can walk, eat well, and reconnect.",
            worries="Does not want anyone to quietly agree to something they cannot comfortably afford.",
            hard_constraints=["Needs the final plan to feel emotionally and financially comfortable for the full group"],
        ),
        TripFriendPersona(
            name="Marco",
            traits=["budget-conscious", "practical", "funny when stressed"],
            budget_notes="Needs flights and hotel costs to stay pretty reasonable.",
            travel_hopes="Would still love a memorable trip if the group can keep the spending under control.",
            worries="Gets nervous about vague plans and hidden costs.",
            hard_constraints=["Would rather skip the trip than commit to something that feels financially reckless"],
        ),
        TripFriendPersona(
            name="Leah",
            traits=["enthusiastic", "warm", "easily inspired by food and scenery"],
            budget_notes="Can stretch a little for the right destination, but not if the group starts getting tense about money.",
            travel_hopes="Wants somewhere beautiful, lively, and easy to enjoy without overplanning every hour.",
            worries="Does not want the chat to turn into a spreadsheet argument.",
            hard_constraints=["Prefers a destination that still feels fun without expensive activities every day"],
        ),
        TripFriendPersona(
            name="Owen",
            traits=["anxious planner", "detail-oriented", "very loyal"],
            budget_notes="Needs enough notice and predictable costs to feel good about going.",
            travel_hopes="Would enjoy traveling if the destination feels logistically simple.",
            worries="Worries the group will choose a place that sounds amazing but becomes exhausting to organize.",
            hard_constraints=["Needs a destination with straightforward flights and a realistic lodging plan"],
        ),
    ],
)

pprint(config)

FriendsTripConfig(admin_name='Trip Host',
                  group_title='Friends Trip Planning',
                  destination_options=['Lisbon', 'Mexico City', 'Vancouver'],
                  friends=[TripFriendPersona(name='Nina',
                                             traits=['empathetic',
                                                     'group glue',
                                                     'likes slow mornings'],
                                             budget_notes='Can travel, but '
                                                          'only if the plan '
                                                          'still feels fair to '
                                                          'friends with '
                                                          'tighter budgets.',
                                             travel_hopes='Wants a place where '
                                                          'everyone can walk, '
       

In [3]:
result = engine.run(config)

name_by_id = {result.admin_member["id"]: result.admin_member["display_name"]}
name_by_id.update({friend["id"]: friend["display_name"] for friend in result.friends})

group_messages = gateway.list_conversation_messages(result.group_conversation["id"])
group_transcript = [
    f"{name_by_id.get(message['sender_id'], message['sender_id'])}: {message['content']}"
    for message in group_messages
]

pprint({
    "final_choice": result.final_choice,
    "consensus_reached": result.consensus_reached,
    "preferences_by_round": result.preferences_by_round,
})

{'consensus_reached': True,
 'final_choice': 'Lisbon',
 'preferences_by_round': [{'Leah': 'Lisbon',
                           'Marco': 'Lisbon',
                           'Nina': 'Lisbon',
                           'Owen': 'Vancouver'},
                          {'Leah': 'Lisbon',
                           'Marco': 'Lisbon',
                           'Nina': 'Lisbon',
                           'Owen': 'Lisbon'}]}


In [4]:
print("Group transcript")
for line in group_transcript:
    print("-", line)

print("\nPrivate brief shown to Marco")
marco_private = gateway.list_member_visible_messages(
    next(friend["id"] for friend in result.friends if friend["display_name"] == "Marco"),
    result.private_conversations["Marco"]["id"],
)
for message in marco_private:
    print("-", message["content"])

Group transcript
- Nina: Hey everyone, can we finally plan a friends trip and see if there is somewhere we can all actually agree on?
- Nina: I'm so down for a trip, but before we start throwing out destinations, can we talk about budget and what everyone's comfortable spending? I want to make sure we pick a place that works for everyone, not just a few of us.
- Marco: I'm so with you on that, let's get a sense of what everyone's comfortable spending before we start dreaming up destinations - I'd say my max for flights and hotel is around $1,000, what about everyone else?
- Leah: I'm thinking $1,000 sounds like a good benchmark for me too, maybe a bit more if we find a destination that's really special - what if we aim for somewhere with plenty of free or low-cost activities so we can relax and not feel like we need to splurge every day?
- Owen: I'm thinking $1,000 sounds doable for me too, but I'd love to get a sense of how much notice we're all thinking we'll need to give ourselves t

## Notes

- This demo uses the same chat interface as the rest of the project through `TestClientChatGateway`.
- The friends only act through chat-visible context plus their own private brief.
- The engine finishes with either a destination or `NO_TRIP` if the group cannot align strongly enough.

In [5]:
engine.close()
client_manager.__exit__(None, None, None)
app.dependency_overrides.clear()
websocket_module.SessionLocal = original_session_local
workspace.cleanup()
print("Cleaned up notebook resources.")

Cleaned up notebook resources.
